# HARUM NOIR — Colab Video Lab

Notebook operacional para gerar vídeos mantendo a **mesma Harum Noir**.

## Regra absoluta
O arquivo incorporado abaixo é o **FACE LOCK**. Todo vídeo oficial deve partir dele.

### Rotas do laboratório
1. **LTX-Video 2B** — principal rota open-source para cenas gerativas com a Noir.
2. **LivePortrait** — melhor para closes/fala/microexpressão usando um driving video.
3. **Stable Video Diffusion XT** — fallback rápido para micro-movimento quando a GPU for limitada.

> Use GPU em `Ambiente de execução > Alterar tipo de ambiente de execução > GPU`.


In [ ]:
# Diagnóstico da GPU
import os, subprocess, json, textwrap, sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM aproximada:", round(props.total_memory/1024**3, 1), "GB")
else:
    print("Ative uma GPU antes de seguir.")


## 1. Restaurar o FACE LOCK canônico

A imagem está embutida no notebook para impedir troca acidental de personagem.


In [ ]:
import base64
from pathlib import Path

FACE_LOCK = Path("/content/HARUM_NOIR_FACE_LOCK_CLEAN_v1.png")
FACE_LOCK.write_bytes(base64.b64decode('...'))
print("FACE LOCK:", FACE_LOCK, FACE_LOCK.stat().st_size, "bytes")


In [ ]:
from PIL import Image
from IPython.display import display
img = Image.open(FACE_LOCK).convert("RGB")
display(img)


# ROTA A — LTX-Video 2B (recomendada)

**Uso:** rotina, ambiente, ações silenciosas, espelho, desenho, café, parede do ateliê.

O LTX oficial suporta image-to-video e o modelo 2B distilled foi escolhido porque exige menos VRAM que o 13B. O mesmo FACE LOCK é usado como quadro de condicionamento inicial.

### Prompt canônico
- sempre começar pela ação;
- dizer que é **a mesma mulher da imagem de referência**;
- cabelo chanel preto imutável;
- figurino preto/grafite;
- luz lateral;
- nada de influencer/locutora.


In [ ]:
# Instalação LTX oficial
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!rm -rf /content/LTX-Video
!git clone -q --depth 1 https://github.com/Lightricks/LTX-Video.git /content/LTX-Video
%cd /content/LTX-Video
!pip -q install -e ".[inference]"
print("LTX instalado.")


In [ ]:
# Biblioteca de cenas Harum Noir
NOIR_SCENES = {
    "espelho": (
        "The same adult woman from the conditioning image stands alone before a slightly imperfect mirror in a quiet night atelier. "
        "Her exact face and black chin-length bob haircut remain unchanged. She studies her reflection, lowers her eyes, then touches a charcoal stick to heavyweight ivory paper. "
        "Minimal matte-black clothing, one tiny aged-gold detail. Carbon black, graphite, ivory and aged gold. "
        "Quiet documentary intimacy, subtle natural breathing, restrained gestures, 50mm cinematic feeling, soft lateral chiaroscuro, shallow depth of field. "
        "No influencer posing, no glamour, no face change, no extra people, no text."
    ),
    "maos_sujas": (
        "The same Harum Noir woman from the reference image sits at a dark wooden drawing table in her night atelier. "
        "Her exact face and black chin-length bob remain unchanged. She rubs charcoal dust between her fingers, makes fast loose anatomical marks on ivory paper, pauses, erases one line, then continues. "
        "Close details of charcoal-stained fingertips, paper texture, quiet breath, black coffee nearby, books and sculpture fragments in soft shadow. "
        "Matte black clothing, graphite apron, subtle aged gold, cinematic realism, intimate handheld micro-movement. "
        "No face drift, no presenter gestures, no bright colors, no logos, no text."
    ),
    "parede": (
        "The same Harum Noir woman from the reference image lifts a finished charcoal study and pins it to a dark atelier wall. "
        "Her exact facial identity and black chin-length bob remain unchanged. She steps back, studies the drawing, then turns slightly toward the mirror without looking at the camera. "
        "Quiet night ambience, carbon and graphite shadows, ivory paper, restrained aged-gold detail, subtle city light through a window. "
        "Poetic documentary cinematography, slow natural motion, 50mm lens feeling, realistic skin and fabric texture. "
        "No face change, no influencer smile, no corporate styling, no text."
    ),
}
for k in NOIR_SCENES:
    print("-", k)


In [ ]:
# Gerar 1 clip LTX (vertical)
import os, glob, subprocess, shlex, time
from pathlib import Path

SCENE = "espelho"           # espelho | maos_sujas | parede
SEED = 230917
HEIGHT = 896                # divisível por 32
WIDTH = 512                 # divisível por 32
NUM_FRAMES = 65             # 8*n + 1; ~2,2 s a 30 fps
OUT_DIR = Path("/content/noir_ltx_outputs")
OUT_DIR.mkdir(exist_ok=True)

prompt = NOIR_SCENES[SCENE]

cmd = [
    "python", "inference.py",
    "--prompt", prompt,
    "--conditioning_media_paths", str(FACE_LOCK),
    "--conditioning_start_frames", "0",
    "--conditioning_strengths", "1.0",
    "--height", str(HEIGHT),
    "--width", str(WIDTH),
    "--num_frames", str(NUM_FRAMES),
    "--seed", str(SEED),
    "--pipeline_config", "configs/ltxv-2b-0.9.8-distilled.yaml",
    "--output_path", str(OUT_DIR),
    "--offload_to_cpu", "True",
]
print("Executando:", " ".join(shlex.quote(x) for x in cmd[:6]), "...")
subprocess.run(cmd, check=True)

videos = sorted(OUT_DIR.glob("*.mp4"), key=lambda p:p.stat().st_mtime, reverse=True)
print("Último vídeo:", videos[0] if videos else "nenhum")


In [ ]:
# Mostrar o último vídeo
from IPython.display import Video, display
if videos:
    display(Video(str(videos[0]), embed=True))


# ROTA B — LivePortrait (FACE LOCK máximo)

**Uso:** fala baixa, olhar, respiração, microexpressões e closes.  
Ele não inventa uma nova pessoa: anima a imagem da Noir a partir de um **driving video**.

Para melhores resultados, o driving video deve ter:
- rosto frontal no primeiro quadro;
- pouco movimento de ombro;
- enquadramento 1:1 ou auto-crop;
- movimentos naturais e pequenos.

Faça upload de um vídeo curto em `/content/driving.mp4`.


In [ ]:
# Instalar LivePortrait oficial
%cd /content
!rm -rf /content/LivePortrait
!git clone -q --depth 1 https://github.com/KwaiVGI/LivePortrait.git
%cd /content/LivePortrait
!pip -q install -r requirements.txt
!git lfs install
!rm -rf temp_pretrained_weights
!git clone -q https://huggingface.co/KwaiVGI/LivePortrait temp_pretrained_weights
!mkdir -p pretrained_weights
!cp -r temp_pretrained_weights/* pretrained_weights/
!rm -rf temp_pretrained_weights
print("LivePortrait instalado.")


In [ ]:
# Se você já subiu /content/driving.mp4, execute:
from pathlib import Path
import subprocess

driving = Path("/content/driving.mp4")
if driving.exists():
    subprocess.run([
        "python", "inference.py",
        "-s", "/content/HARUM_NOIR_FACE_LOCK_CLEAN_v1.png",
        "-d", "/content/driving.mp4",
        "--flag_crop_driving_video",
    ], check=True)
    print("Procure o resultado em /content/LivePortrait/animations/")
else:
    print("Envie um driving video para /content/driving.mp4 antes de executar.")


# ROTA C — Stable Video Diffusion XT (fallback rápido)

**Uso:** micro-movimento e retrato vivo quando o LTX estiver pesado demais.  
Não entende roteiro textual; serve para respiração, pequenos movimentos e transições.


In [ ]:
%cd /content
!pip -q install -U diffusers transformers accelerate imageio[ffmpeg]
import torch
from PIL import Image, ImageOps
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video

svd = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16",
)
svd.enable_model_cpu_offload()
svd.unet.enable_forward_chunking()

source = Image.open(FACE_LOCK).convert("RGB")
# SVD foi treinado em 1024x576; usamos retrato vertical preservando o rosto.
source = ImageOps.fit(source, (576, 1024), method=Image.Resampling.LANCZOS)

gen = torch.manual_seed(230917)
frames = svd(
    source,
    decode_chunk_size=4,
    generator=gen,
    motion_bucket_id=35,
    noise_aug_strength=0.015,
).frames[0]

svd_out = "/content/HARUM_NOIR_SVD_MICROMOTION.mp4"
export_to_video(frames, svd_out, fps=6)
print(svd_out)


In [ ]:
from IPython.display import Video, display
display(Video("/content/HARUM_NOIR_SVD_MICROMOTION.mp4", embed=True))


# Política de publicação

Um vídeo só vira **Harum Noir oficial** se:
1. rosto = FACE LOCK;
2. cabelo = mesmo chanel;
3. roupa = preto/grafite;
4. paleta = carbono/grafite/marfim/ouro;
5. movimento = natural e contido;
6. nenhum fato biográfico inventado;
7. existe novidade narrativa;
8. CTA corresponde ao estado real do produto.

Salve os aprovados em uma pasta do Drive chamada `HARUM_NOIR/VIDEO_APPROVED`.
